# 80: STH-MVRV Zone-Based Exits

**Insight from Notebook 79:** Danger zone exits never triggered (score ≥0.60 too conservative). Need a more sensitive exit signal.

## The Hypothesis:

STH-MVRV (Short-Term Holder MVRV) is Check's "Swiss Army Knife" indicator that measures short-term holder profitability:

| STH-MVRV | Zone | Meaning | Action |
|----------|------|---------|--------|
| < 0.90 | **Deep Cooled** | STH underwater, capitulation | **STRONG BUY** |
| 0.90-1.00 | **Cooled** | STH near breakeven, support | **BUY** |
| 1.00-1.30 | **Fair Value** | Healthy profit-taking | **HOLD** |
| 1.30-1.60 | **Warming** | STH taking profits, caution | **HOLD/TRIM** |
| 1.60-1.90 | **Local Top** | High profit-taking pressure | **CONSIDER EXIT** |
| > 1.90 | **Overheated** | Extreme euphoria, distribution | **EXIT** |

## Why STH-MVRV for Exits?

1. **Available for full history** (2011+) - no funding data needed
2. **Part of Check's framework** - "Swiss Army Knife" indicator
3. **Directly measures sell pressure** - high STH-MVRV = profitable holders selling
4. **Mean-reverting** - always returns to equilibrium
5. **Works across all regimes** - not regime-dependent

## What We'll Test:

**Entry:** Buy The Dip (4/5 conditions) - stays the same

**Exit strategies:**
1. Never Exit (baseline)
2. STH-MVRV > 1.5 (Warming)
3. STH-MVRV > 1.7 (Local Top forming)
4. STH-MVRV > 1.9 (Overheated)
5. STH-MVRV > 2.1 (Extreme euphoria)
6. Original (MVRV>2.0 AND LTH-SOPR>1.5)

**Expected:** STH-MVRV 1.7-1.9 range should capture local tops while avoiding whipsaws.

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    import vectorbt as vbt
    print("✓ VectorBT loaded")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "vectorbt"])
    import vectorbt as vbt

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Setup complete")

In [ ]:
# Configuration
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "brk" / "daily"
GLASSNODE_DIR = PROJECT_ROOT / "data" / "glassnode" / "daily"

FEES = 0.001
SLIPPAGE = 0.001

## 1. Load Data

In [ ]:
def load_metric(name: str, source: str = "brk") -> pd.Series:
    path = DATA_DIR / f"{name}.parquet" if source == "brk" else GLASSNODE_DIR / f"{name}.parquet"
    if not path.exists():
        return pd.Series(dtype=float)
    df = pd.read_parquet(path)
    if 'time' not in df.columns and isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        if len(df.columns) == 2:
            df.columns = ['time', 'value']
    if 'value' not in df.columns:
        for col in df.columns:
            if col != 'time' and pd.api.types.is_numeric_dtype(df[col]):
                df['value'] = df[col]
                break
    if 'time' in df.columns and 'value' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        return df.set_index('time')['value'].sort_index()
    return pd.Series(dtype=float)

print("Loading data...")

metrics = {
    'price': ('price', 'brk'),
    'mvrv': ('mvrv', 'brk'),
    'mvrv_sth': ('mvrv_sth', 'brk'),
    'sopr_sth': ('sopr_sth', 'brk'),
    'sopr_lth': ('sopr_lth', 'brk'),
    'realized_profit': ('realized_profit', 'brk'),
    'realized_loss': ('realized_loss', 'brk'),
    'funding': ('funding_rate', 'glassnode'),
    'liq_long': ('liquidations_long', 'glassnode'),
    'liq_short': ('liquidations_short', 'glassnode'),
}

df_dict = {name: load_metric(metric, source) for name, (metric, source) in metrics.items()}
df = pd.DataFrame(df_dict).fillna(method='ffill')

print(f"✓ Data: {len(df)} days ({df.index[0].date()} to {df.index[-1].date()})")
print(f"  STH-MVRV available: {df['mvrv_sth'].notna().sum()} days")
print(f"  STH-MVRV range: {df['mvrv_sth'].min():.2f} to {df['mvrv_sth'].max():.2f}")
df.head()

## 2. Analyze STH-MVRV Distribution

In [ ]:
# STH-MVRV statistics
sth_mvrv = df['mvrv_sth'].dropna()

print("STH-MVRV Statistics:")
print("="*70)
print(f"Mean:     {sth_mvrv.mean():.3f}")
print(f"Median:   {sth_mvrv.median():.3f}")
print(f"Std Dev:  {sth_mvrv.std():.3f}")
print(f"Min:      {sth_mvrv.min():.3f}")
print(f"Max:      {sth_mvrv.max():.3f}")

print("\nPercentiles:")
for p in [5, 10, 25, 50, 75, 90, 95, 99]:
    val = sth_mvrv.quantile(p/100)
    print(f"  {p}th: {val:.3f}")

# Zone distribution
print("\nTime Spent in Each Zone:")
print("="*70)
zones = [
    ('Deep Cooled (<0.90)', sth_mvrv < 0.90),
    ('Cooled (0.90-1.00)', (sth_mvrv >= 0.90) & (sth_mvrv < 1.00)),
    ('Fair Value (1.00-1.30)', (sth_mvrv >= 1.00) & (sth_mvrv < 1.30)),
    ('Warming (1.30-1.60)', (sth_mvrv >= 1.30) & (sth_mvrv < 1.60)),
    ('Local Top (1.60-1.90)', (sth_mvrv >= 1.60) & (sth_mvrv < 1.90)),
    ('Overheated (>1.90)', sth_mvrv >= 1.90),
]

for zone_name, mask in zones:
    days = mask.sum()
    pct = days / len(sth_mvrv) * 100
    print(f"{zone_name:<30} {days:>5} days ({pct:>5.1f}%)")

## 3. Visualize STH-MVRV Zones

In [ ]:
# Plot STH-MVRV zones
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Price with zones
ax1 = axes[0]
ax1.plot(df.index, df['price'], color='black', linewidth=2, label='BTC Price')

# Color zones
ax1.fill_between(df.index, 0, df['price'].max() * 1.2,
                  where=(df['mvrv_sth'] >= 1.90), alpha=0.3, color='red', label='Overheated (>1.90)')
ax1.fill_between(df.index, 0, df['price'].max() * 1.2,
                  where=(df['mvrv_sth'] >= 1.60) & (df['mvrv_sth'] < 1.90), alpha=0.25, color='orange', label='Local Top (1.60-1.90)')
ax1.fill_between(df.index, 0, df['price'].max() * 1.2,
                  where=(df['mvrv_sth'] >= 1.30) & (df['mvrv_sth'] < 1.60), alpha=0.2, color='yellow', label='Warming (1.30-1.60)')
ax1.fill_between(df.index, 0, df['price'].max() * 1.2,
                  where=(df['mvrv_sth'] < 0.90), alpha=0.2, color='green', label='Deep Cooled (<0.90)')

ax1.set_ylabel('BTC Price ($)', fontsize=12)
ax1.set_title('STH-MVRV Zones Over Time', fontsize=14, fontweight='bold')
ax1.set_yscale('log')
ax1.legend(fontsize=10, loc='upper left')
ax1.grid(True, alpha=0.3)

# STH-MVRV value
ax2 = axes[1]
ax2.plot(df.index, df['mvrv_sth'], color='blue', linewidth=1.5, label='STH-MVRV')
ax2.axhline(1.0, color='gray', linestyle='-', alpha=0.5, label='Equilibrium (1.0)')
ax2.axhline(1.5, color='yellow', linestyle='--', alpha=0.7, label='Warming (1.5)')
ax2.axhline(1.7, color='orange', linestyle='--', alpha=0.7, label='Local Top (1.7)')
ax2.axhline(1.9, color='red', linestyle='--', alpha=0.7, label='Overheated (1.9)')
ax2.fill_between(df.index, 0, 3, where=(df['mvrv_sth'] >= 1.9), alpha=0.2, color='red')
ax2.fill_between(df.index, 0, 3, where=(df['mvrv_sth'] < 0.9), alpha=0.2, color='green')
ax2.set_ylabel('STH-MVRV', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylim(0.5, 2.5)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Green = Deep buying opportunity (STH underwater)")
print("📊 Yellow = Warming (caution, profit-taking increasing)")
print("📊 Orange = Local top forming (high profit-taking)")
print("📊 Red = Overheated (extreme euphoria, exit zone)")

## 4. Generate Entry Signals

In [ ]:
# Entry: Buy The Dip (4/5 conditions)
print("Generating entry signals...\n")

c1 = df['mvrv_sth'] < 1.0
c2 = df['sopr_sth'] < 1.0
c3 = (df['realized_profit'] / df['realized_loss']) < 1.0
c4 = df['funding'] <= 0.0
c5 = (df['liq_long'] / df['liq_short']) > 1.0

# Handle NaN values
c4 = c4.fillna(False)
c5 = c5.fillna(False)

entry_count = c1.astype(int) + c2.astype(int) + c3.astype(int) + c4.astype(int) + c5.astype(int)
entries = (entry_count >= 4).fillna(False).astype(bool)

print(f"✓ Entry signals: {entries.sum()}")
print(f"  Entries dtype: {entries.dtype}")

## 5. Generate Exit Signals (STH-MVRV Thresholds)

In [ ]:
print("Generating exit signals...\n")

# Test different STH-MVRV thresholds
exit_strategies = {
    'never': (pd.Series(False, index=df.index, dtype=bool), 'Never Exit'),
    'sth_1_3': ((df['mvrv_sth'] > 1.3).fillna(False).astype(bool), 'STH-MVRV > 1.3 (Warming start)'),
    'sth_1_5': ((df['mvrv_sth'] > 1.5).fillna(False).astype(bool), 'STH-MVRV > 1.5 (Warming)'),
    'sth_1_7': ((df['mvrv_sth'] > 1.7).fillna(False).astype(bool), 'STH-MVRV > 1.7 (Local Top)'),
    'sth_1_9': ((df['mvrv_sth'] > 1.9).fillna(False).astype(bool), 'STH-MVRV > 1.9 (Overheated)'),
    'sth_2_1': ((df['mvrv_sth'] > 2.1).fillna(False).astype(bool), 'STH-MVRV > 2.1 (Extreme)'),
    'original': (((df['mvrv'] > 2.0) & (df['sopr_lth'] > 1.5)).fillna(False).astype(bool), 'Original (MVRV>2.0)'),
}

print("Exit signal counts:")
print("="*70)
for key, (exits, name) in exit_strategies.items():
    print(f"{name:<40} {exits.sum():>4} signals")

# Verify dtypes
print("\nDtype verification:")
for key, (exits, name) in exit_strategies.items():
    print(f"  {key}: {exits.dtype}")

## 6. Backtest All Strategies

In [ ]:
def backtest_strategy(df: pd.DataFrame, entries: pd.Series, exits: pd.Series, name: str) -> dict:
    """Run backtest using VectorBT."""
    pf = vbt.Portfolio.from_signals(
        close=df['price'],
        entries=entries,
        exits=exits,
        fees=FEES,
        slippage=SLIPPAGE,
        init_cash=10000,
        freq='1D'
    )
    return {
        'name': name,
        'portfolio': pf,
        'total_return': pf.total_return() * 100,
        'sharpe': pf.sharpe_ratio(),
        'max_dd': pf.max_drawdown() * 100,
        'num_trades': pf.trades.count(),
        'win_rate': pf.trades.win_rate() * 100 if pf.trades.count() > 0 else 0,
    }

print("\n" + "="*90)
print("BACKTESTING: STH-MVRV ZONE-BASED EXITS")
print("="*90)

# Backtest all strategies
results = {}
for key, (exits, name) in exit_strategies.items():
    results[key] = backtest_strategy(df, entries, exits, name)

# Buy and hold
bh_return = (df['price'].iloc[-1] / df['price'].iloc[0] - 1) * 100

# Results table
print(f"\n{'Strategy':<40} {'Return':>12} {'Sharpe':>8} {'Max DD':>10} {'Trades':>8} {'Win Rate':>10}")
print("-"*90)

for key, res in results.items():
    print(f"{res['name']:<40} {res['total_return']:>11.1f}% {res['sharpe']:>8.2f} {res['max_dd']:>9.1f}% {int(res['num_trades']):>8} {res['win_rate']:>9.1f}%")

print(f"{'Buy & Hold':<40} {bh_return:>11.1f}% {'~1.0':>8} {'?':>10} {'-':>8} {'-':>10}")
print("="*90)

# Comparison
print("\nVS BUY & HOLD:")
for key, res in results.items():
    diff = res['total_return'] - bh_return
    status = "✅ BEAT" if diff > 0 else "❌ LOST"
    print(f"  {res['name']:<40} {status} by {abs(diff):.1f}%")

# Best strategy
best = max(results.values(), key=lambda x: x['total_return'])
print(f"\n🏆 BEST STRATEGY: {best['name']} at {best['total_return']:.1f}%")

# Best Sharpe
best_sharpe = max(results.values(), key=lambda x: x['sharpe'])
print(f"📊 BEST SHARPE: {best_sharpe['name']} at {best_sharpe['sharpe']:.2f}")

## 7. Equity Curves Comparison

In [ ]:
# Plot equity curves
fig, ax = plt.subplots(figsize=(16, 8))

# Buy & Hold
bh_equity = (df['price'] / df['price'].iloc[0]) * 10000
ax.plot(bh_equity.index, bh_equity.values, label='Buy & Hold', linewidth=3, color='black', linestyle='--', alpha=0.7)

# Strategies
colors = {'never': 'green', 'sth_1_3': 'purple', 'sth_1_5': 'blue', 'sth_1_7': 'orange', 'sth_1_9': 'red', 'sth_2_1': 'darkred', 'original': 'gray'}
for key, res in results.items():
    equity = res['portfolio'].value()
    ax.plot(equity.index, equity.values, label=res['name'], linewidth=2, alpha=0.8, color=colors.get(key, 'gray'))

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Portfolio Value ($)', fontsize=12)
ax.set_title('STH-MVRV Zone-Based Exits: Equity Curves', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal Portfolio Values:")
print(f"  Buy & Hold: ${bh_equity.iloc[-1]:,.0f}")
for key, res in results.items():
    final = res['portfolio'].value().iloc[-1]
    print(f"  {res['name']}: ${final:,.0f}")

## 8. Analyze Exit Timing

In [ ]:
# Analyze when exits fired and subsequent price action
print("\n" + "="*90)
print("EXIT TIMING ANALYSIS")
print("="*90)

# Test different thresholds
for key in ['sth_1_5', 'sth_1_7', 'sth_1_9']:
    exits, name = exit_strategies[key]
    if exits.sum() == 0:
        print(f"\n{name}: No exit signals")
        continue
    
    print(f"\n{name}:")
    print("-" * 70)
    
    # Get exit dates
    exit_dates = df[exits].index
    
    # Analyze price action after exits
    forward_returns = []
    for exit_date in exit_dates[:10]:  # Show first 10 exits
        exit_price = df.loc[exit_date, 'price']
        
        # Forward returns (30, 60, 90 days)
        try:
            future_30d = df[df.index > exit_date].iloc[30]['price'] if len(df[df.index > exit_date]) > 30 else None
            future_60d = df[df.index > exit_date].iloc[60]['price'] if len(df[df.index > exit_date]) > 60 else None
            future_90d = df[df.index > exit_date].iloc[90]['price'] if len(df[df.index > exit_date]) > 90 else None
            
            ret_30d = (future_30d / exit_price - 1) * 100 if future_30d else None
            ret_60d = (future_60d / exit_price - 1) * 100 if future_60d else None
            ret_90d = (future_90d / exit_price - 1) * 100 if future_90d else None
            
            sth_val = df.loc[exit_date, 'mvrv_sth']
            
            print(f"  {exit_date.date()}: ${exit_price:,.0f} (STH-MVRV: {sth_val:.2f})")
            if ret_30d:
                print(f"    30d: {ret_30d:+.1f}% | 60d: {ret_60d:+.1f}% | 90d: {ret_90d:+.1f}%")
        except:
            pass
    
    if len(exit_dates) > 10:
        print(f"  ... and {len(exit_dates) - 10} more exit signals")

print("\n" + "="*90)

## 9. Performance by Time Period

In [ ]:
# Split by major periods
periods = [
    ('2013-01-01', '2015-12-31', '2013-2015 (Early)'),
    ('2017-01-01', '2018-12-31', '2017-2018 Cycle'),
    ('2020-01-01', '2022-12-31', '2020-2022 Cycle'),
    ('2023-01-01', '2026-01-22', '2023-2026 Bull'),
]

print("\n" + "="*100)
print("PERFORMANCE BY TIME PERIOD")
print("="*100)

for start, end, label in periods:
    period_df = df[(df.index >= start) & (df.index <= end)].copy()
    if len(period_df) < 30:
        continue
    
    period_entries = entries[(entries.index >= start) & (entries.index <= end)]
    
    # Buy and hold
    bh = (period_df['price'].iloc[-1] / period_df['price'].iloc[0] - 1) * 100
    
    # Test STH-MVRV strategies
    period_results = {}
    for key in ['never', 'sth_1_5', 'sth_1_7', 'sth_1_9', 'original']:
        period_exits = exit_strategies[key][0][(exit_strategies[key][0].index >= start) & (exit_strategies[key][0].index <= end)]
        
        try:
            pf = vbt.Portfolio.from_signals(
                close=period_df['price'], entries=period_entries, exits=period_exits,
                fees=FEES, slippage=SLIPPAGE, init_cash=10000, freq='1D'
            )
            period_results[key] = pf.total_return() * 100
        except:
            period_results[key] = 0
    
    # STH-MVRV stats for period
    period_sth = period_df['mvrv_sth'].dropna()
    overheated_days = (period_sth > 1.9).sum()
    warming_days = ((period_sth > 1.5) & (period_sth <= 1.9)).sum()
    
    print(f"\n{label}:")
    print(f"  STH-MVRV > 1.9 (Overheated): {overheated_days} days ({overheated_days / len(period_sth) * 100:.1f}%)")
    print(f"  STH-MVRV > 1.5 (Warming): {warming_days} days ({warming_days / len(period_sth) * 100:.1f}%)")
    print(f"  Buy & Hold: {bh:+.1f}%")
    print(f"  Never Exit: {period_results['never']:+.1f}% ({period_results['never'] - bh:+.1f}% vs B&H)")
    print(f"  STH-MVRV > 1.5: {period_results['sth_1_5']:+.1f}% ({period_results['sth_1_5'] - bh:+.1f}% vs B&H)")
    print(f"  STH-MVRV > 1.7: {period_results['sth_1_7']:+.1f}% ({period_results['sth_1_7'] - bh:+.1f}% vs B&H)")
    print(f"  STH-MVRV > 1.9: {period_results['sth_1_9']:+.1f}% ({period_results['sth_1_9'] - bh:+.1f}% vs B&H)")
    print(f"  Original: {period_results['original']:+.1f}% ({period_results['original'] - bh:+.1f}% vs B&H)")
    
    # Winner
    best_key = max(period_results, key=period_results.get)
    print(f"  🏆 Winner: {exit_strategies[best_key][1]}")

print("\n" + "="*100)

## 10. Final Verdict

In [ ]:
print("\n" + "="*90)
print("FINAL VERDICT: STH-MVRV ZONE-BASED EXITS")
print("="*90)

never = results['never']
sth_1_5 = results['sth_1_5']
sth_1_7 = results['sth_1_7']
sth_1_9 = results['sth_1_9']
original = results['original']
best = max(results.values(), key=lambda x: x['total_return'])
best_sharpe = max(results.values(), key=lambda x: x['sharpe'])

print(f"\n1. PERFORMANCE COMPARISON:")
print(f"   Buy & Hold:       {bh_return:.1f}%")
print(f"   Never Exit:       {never['total_return']:.1f}% ({never['total_return'] - bh_return:+.1f}%)")
print(f"   STH-MVRV > 1.5:   {sth_1_5['total_return']:.1f}% ({sth_1_5['total_return'] - bh_return:+.1f}%)")
print(f"   STH-MVRV > 1.7:   {sth_1_7['total_return']:.1f}% ({sth_1_7['total_return'] - bh_return:+.1f}%)")
print(f"   STH-MVRV > 1.9:   {sth_1_9['total_return']:.1f}% ({sth_1_9['total_return'] - bh_return:+.1f}%)")
print(f"   Original:         {original['total_return']:.1f}% ({original['total_return'] - bh_return:+.1f}%)")
print(f"   Best Return: {best['name']} at {best['total_return']:.1f}%")

print(f"\n2. RISK-ADJUSTED PERFORMANCE:")
print(f"   Never Exit:     Sharpe {never['sharpe']:.2f}, DD {never['max_dd']:.1f}%")
print(f"   STH-MVRV > 1.5: Sharpe {sth_1_5['sharpe']:.2f}, DD {sth_1_5['max_dd']:.1f}%")
print(f"   STH-MVRV > 1.7: Sharpe {sth_1_7['sharpe']:.2f}, DD {sth_1_7['max_dd']:.1f}%")
print(f"   STH-MVRV > 1.9: Sharpe {sth_1_9['sharpe']:.2f}, DD {sth_1_9['max_dd']:.1f}%")
print(f"   Original:       Sharpe {original['sharpe']:.2f}, DD {original['max_dd']:.1f}%")
print(f"   Best Sharpe: {best_sharpe['name']} at {best_sharpe['sharpe']:.2f}")

print(f"\n3. TRADE ACTIVITY:")
print(f"   Never Exit:     {int(never['num_trades'])} trades, {never['win_rate']:.1f}% win rate")
print(f"   STH-MVRV > 1.5: {int(sth_1_5['num_trades'])} trades, {sth_1_5['win_rate']:.1f}% win rate")
print(f"   STH-MVRV > 1.7: {int(sth_1_7['num_trades'])} trades, {sth_1_7['win_rate']:.1f}% win rate")
print(f"   STH-MVRV > 1.9: {int(sth_1_9['num_trades'])} trades, {sth_1_9['win_rate']:.1f}% win rate")
print(f"   Original:       {int(original['num_trades'])} trades, {original['win_rate']:.1f}% win rate")

# Analysis
improvement_1_7 = sth_1_7['total_return'] - never['total_return']
improvement_1_9 = sth_1_9['total_return'] - never['total_return']

print(f"\n4. KEY INSIGHTS:")
print(f"   STH-MVRV > 1.7 vs Never Exit: {improvement_1_7:+.1f}%")
print(f"   STH-MVRV > 1.9 vs Never Exit: {improvement_1_9:+.1f}%")

if improvement_1_7 > 20 or improvement_1_9 > 20:
    print(f"   ✅ STH-MVRV exits add SIGNIFICANT value!")
elif improvement_1_7 > 5 or improvement_1_9 > 5:
    print(f"   ✓ STH-MVRV exits add modest value")
else:
    print(f"   ❌ STH-MVRV exits don't improve performance")

print(f"\n5. RECOMMENDATION:")

if best['name'] == 'Never Exit':
    print(f"   🎯 Optimal Strategy: Use Check's entries, NEVER EXIT")
    print(f"   📝 STH-MVRV exits don't improve returns")
    if best_sharpe['name'] != 'Never Exit':
        print(f"   💡 BUT {best_sharpe['name']} has better risk-adjusted returns (Sharpe {best_sharpe['sharpe']:.2f})")
        print(f"   Consider using {best_sharpe['name']} if you prioritize drawdown protection")
elif best['name'].startswith('STH-MVRV'):
    threshold = best['name'].split('>')[1].split()[0]
    print(f"   🎯 Optimal Strategy: Use Check's entries + STH-MVRV > {threshold} exits")
    print(f"   📝 Exit when short-term holders are highly profitable ({threshold})")
    print(f"   ✅ This captures local tops while staying invested in trends")
else:
    print(f"   🎯 Optimal Strategy: {best['name']}")

# Final assessment
if best['total_return'] > bh_return:
    print(f"\n   🏆 SUCCESS: This strategy BEATS buy-and-hold by {best['total_return'] - bh_return:.1f}%!")
else:
    gap = bh_return - best['total_return']
    print(f"\n   ⚠️  Still trails buy-and-hold by {gap:.1f}%")
    if best_sharpe['sharpe'] > 1.0:
        print(f"   But best risk-adjusted strategy has Sharpe {best_sharpe['sharpe']:.2f}")

print("\n" + "="*90)

## Conclusion

STH-MVRV is Check's "Swiss Army Knife" indicator - it directly measures short-term holder profitability and has historically been excellent at identifying:
- **Buy zones** (STH-MVRV < 0.90): Capitulation, underwater holders
- **Sell zones** (STH-MVRV > 1.7-1.9): Euphoria, profitable holders distributing

**Key Questions Answered:**
1. Does STH-MVRV improve upon "Never Exit"?
2. What threshold works best (1.5, 1.7, 1.9, 2.1)?
3. Does this work in 2017/2021 tops AND 2023-2026 bull?
4. Can we finally beat buy-and-hold?